## ✅ FIXED: JupyterLab Tab Integration (v0.7.2)

### Key Fix Applied:
- **Removed server proxy launcher_entry** that was causing browser tab behavior
- **Frontend extension now handles launcher** exclusively  
- **Added Firefox icon styling** with inline CSS
- **Verified widget creation** for proper JupyterLab tab integration

### What Changed:
1. **Server Proxy**: Removed `launcher_entry` to prevent conflicting launchers
2. **Frontend**: Enhanced with Firefox icon CSS and widget styling  
3. **Tab Behavior**: Now properly opens in JupyterLab tab via `app.shell.add()`

## 🐛 Debug Results: Duplicate Launcher Issue Found

Your server debug output shows the problem:

```
❌ SERVER PROXY LAUNCHER FOUND:
   Title: Firefox Browser
   Path: firefox-desktop
   This creates the launcher in Notebooks section!
```

### Root Cause:
The server is still running **v0.7.1** which has the `launcher_entry` in server proxy configuration. The fix in **v0.7.2** that removes this launcher_entry hasn't been deployed yet.

### Solution:
Deploy the **v0.7.2** wheel that has the `launcher_entry` removed from server proxy.

In [ ]:
#!/bin/bash

# Deploy v0.7.2 to fix the duplicate launcher issue

echo "🚀 Deploying v0.7.2 with launcher_entry removed"

# Step 1: Upload the v0.7.2 wheel (run on local machine)
# scp dist/jupyterlab_firefox_launcher-0.7.2-py3-none-any.whl ubuntu@your-server:/tmp/

# Step 2: Install on server (run on server)
sudo pip install /tmp/jupyterlab_firefox_launcher-0.7.2-py3-none-any.whl --force-reinstall

# Step 3: Restart JupyterHub
sudo systemctl restart jupyterhub

echo "✅ Deployment complete!"
echo ""
echo "🧪 Test after deployment:"
echo "1. Access JupyterLab"
echo "2. Check Launcher - should only see ONE Firefox icon in 'Other' section"
echo "3. No Firefox icon should appear in 'Notebooks' section"

# Step 4: Verify the fix (run on server after restart)
echo ""
echo "🔍 Verify fix with debug script:"
echo "python debug.py"
echo ""
echo "Expected output:"
echo "✅ No server proxy launcher_entry found"

## 🐛 Xpra Startup Issues Analysis

From your server logs, I can see several non-critical warnings but Xpra is still starting successfully:

### ✅ Working (Good Signs):
- `created tcp socket '0.0.0.0:38227'` - TCP binding successful
- `created unix domain socket` - Socket creation working  
- `serving html content from '/usr/share/xpra/www'` - HTML5 client available
- `OpenGL is supported on display ':0'` - Display working
- `pointer device emulation using XTest` - Input working

### ⚠️ Non-Critical Warnings:
1. **XDG_RUNTIME_DIR** - Missing `/run/user/1000/xpra` directory
2. **uinput module** - Python uinput module not available  
3. **dbus-launch** - D-Bus session issues
4. **nodejs warning** - JupyterLab build status (unrelated to Firefox)

### 🎯 Key Point:
Despite the warnings, **Xpra is starting successfully** and Firefox should work!

In [ ]:
#!/bin/bash

# Optional: Reduce Xpra warning messages (these don't affect functionality)

echo "🔧 Optional Xpra Warning Fixes (run on server)"
echo "================================================="

# 1. Fix XDG_RUNTIME_DIR warning
echo "📁 Creating XDG_RUNTIME_DIR for Xpra..."
sudo mkdir -p /run/user/1000/xpra
sudo chown ubuntu:ubuntu /run/user/1000/xpra
export XDG_RUNTIME_DIR=/run/user/1000

# 2. Install uinput module (optional - for better input handling)
echo "🎮 Installing python uinput module..."
sudo apt-get update
sudo apt-get install -y python3-uinput

# 3. Fix D-Bus issues (optional - for better desktop integration)
echo "🔌 Installing D-Bus session components..."
sudo apt-get install -y dbus-x11

echo ""
echo "✅ Warning fixes applied!"
echo ""
echo "📝 Note: These warnings don't prevent Firefox from working."
echo "   Firefox should still launch successfully in JupyterLab tabs."
echo ""
echo "🧪 Test Firefox launcher now:"
echo "   1. Access JupyterLab"
echo "   2. Click Firefox icon in Launcher > Other section"
echo "   3. Should open in JupyterLab tab with fewer warning messages"

# Firefox JupyterLab Tab Integration Test

This notebook demonstrates the key differences between accessing Firefox directly via URL vs. using the JupyterLab extension launcher, and provides steps to ensure Firefox opens in JupyterLab tabs.

## Current Issue Analysis

**Direct URL Access**: `https://bdxbdxbdx-0d317c8b-1cfe-423e-a518-57f97fd50c6e.vantagecompute.ai/user/ubuntu/a/firefox-desktop/`
- ✅ Firefox launches successfully via Xpra HTML5
- ❌ Opens in new browser tab (bypasses JupyterLab extension)

**JupyterLab Extension Access**: Via Firefox launcher icon in JupyterLab
- ✅ Should embed Firefox in JupyterLab iframe widget  
- ✅ Should open in JupyterLab tab, not browser tab

## Section 1: Setup Development Environment

First, let's verify our development environment and extension status.

In [ ]:
#!/usr/bin/env python3
"""
Check JupyterLab extension installation and configuration status
"""

import subprocess
import sys
import json
from pathlib import Path

print("🔍 Checking JupyterLab Firefox Launcher Extension Status")
print("=" * 60)

# Check if extension is installed
try:
    result = subprocess.run(['jupyter', 'labextension', 'list'], 
                          capture_output=True, text=True, check=True)
    
    if 'jupyterlab-firefox-launcher' in result.stdout:
        print("✅ Extension is installed in JupyterLab")
        # Extract version info
        lines = result.stdout.split('\n')
        for line in lines:
            if 'jupyterlab-firefox-launcher' in line:
                print(f"   {line.strip()}")
    else:
        print("❌ Extension NOT found in JupyterLab")
        print("📋 Available extensions:")
        print(result.stdout)
        
except subprocess.CalledProcessError as e:
    print(f"❌ Error checking extensions: {e}")
    print(f"stdout: {e.stdout}")
    print(f"stderr: {e.stderr}")

print("\n🔧 Checking jupyter-server-proxy status...")
try:
    result = subprocess.run(['jupyter', 'server', 'extension', 'list'], 
                          capture_output=True, text=True, check=True)
    
    if 'jupyter_server_proxy' in result.stdout:
        print("✅ jupyter-server-proxy is enabled")
    else:
        print("❌ jupyter-server-proxy not found")
        
except subprocess.CalledProcessError as e:
    print(f"❌ Error checking server extensions: {e}")

## Section 2: Current Extension Package Structure

Let's examine the current extension structure to understand how it should work.

In [ ]:
#!/usr/bin/env python3
"""
Examine the frontend implementation to verify JupyterLab tab integration
"""

import os
from pathlib import Path

print("🔍 Examining Frontend Implementation")
print("=" * 50)

# Check if frontend source exists
frontend_index = Path("frontend/src/index.ts")
if frontend_index.exists():
    print("✅ Frontend source file exists")
    
    # Read and analyze the TypeScript source
    with open(frontend_index, 'r') as f:
        content = f.read()
    
    print("\n📋 Key Implementation Features:")
    
    # Check for proper imports
    if '@jupyterlab/coreutils' in content and 'URLExt' in content:
        print("✅ URLExt import for proper URL construction")
    else:
        print("❌ Missing URLExt import")
    
    if '@lumino/widgets' in content and 'Widget' in content:
        print("✅ Widget import for JupyterLab tab integration")
    else:
        print("❌ Missing Widget import")
    
    # Check for proper URL construction
    if 'URLExt.join(baseUrl, \'firefox-desktop\')' in content:
        print("✅ Proper URL construction with baseUrl")
    else:
        print("❌ Missing proper URL construction")
    
    # Check for JupyterLab tab opening
    if 'app.shell.add(widget, \'main\')' in content:
        print("✅ Opens in JupyterLab tab via app.shell.add()")
    else:
        print("❌ Missing JupyterLab tab integration")
    
    # Check for iframe embedding
    if 'iframe.src = firefoxUrl' in content:
        print("✅ Uses iframe for embedding Firefox")
    else:
        print("❌ Missing iframe implementation")
        
    print(f"\n📄 Frontend file size: {len(content)} characters")
    
else:
    print("❌ Frontend source file not found!")

# Check built JavaScript
lib_index = Path("lib/index.js")
if lib_index.exists():
    print(f"\n✅ Built JavaScript exists ({lib_index.stat().st_size} bytes)")
else:
    print("\n❌ Built JavaScript not found!")

## Section 3: Frontend Widget Configuration

The key difference between browser tab and JupyterLab tab behavior lies in the frontend implementation.

In [ ]:
// Correct Implementation: JupyterLab Tab Integration
// This is what the extension should do vs. direct URL access

// ❌ WRONG: Direct URL access (opens in browser tab)
// window.open('https://your-server/user/ubuntu/a/firefox-desktop/', '_blank');

// ✅ CORRECT: JupyterLab Widget (opens in JupyterLab tab)
const iframe = document.createElement('iframe');
iframe.src = firefoxUrl;  // Proper URL from URLExt.join()
iframe.style.width = '100%';
iframe.style.height = '100%';
iframe.style.border = 'none';
iframe.setAttribute('allowfullscreen', 'true');

const widget = new Widget({ node: iframe });
widget.id = 'firefox-desktop-' + Date.now();
widget.title.label = 'Firefox Desktop';
widget.title.iconClass = 'jp-FirefoxIcon';
widget.title.closable = true;

// The magic: Open in JupyterLab tab, not browser tab
app.shell.add(widget, 'main');
app.shell.activateById(widget.id);

## Section 4: Server Proxy Configuration

Let's verify the server proxy is working correctly (which it is, since Firefox launches successfully).

In [ ]:
#!/usr/bin/env python3
"""
Test server proxy configuration and endpoints
"""

import subprocess
import os

print("🔍 Server Proxy Configuration Test")
print("=" * 50)

# Test the server proxy setup function
try:
    print("✅ Testing firefox-desktop entry point...")
    
    # Import our server proxy module
    import sys
    sys.path.insert(0, 'jupyterlab_firefox_launcher')
    
    from jupyterlab_firefox_launcher.server_proxy import setup_firefox_desktop
    
    # Get the configuration
    config = setup_firefox_desktop()
    
    print("📋 Server Proxy Configuration:")
    print(f"   Command: {' '.join(config['command'][:3])}...")  # Show first few parts
    print(f"   Port: {config.get('port', 'auto-assigned')}")
    print(f"   Timeout: {config.get('timeout', 'default')}")
    print(f"   New Browser Window: {config.get('new_browser_window', 'undefined')}")
    
    # Check launcher entry
    launcher = config.get('launcher_entry', {})
    if launcher:
        print(f"   Launcher Title: {launcher.get('title', 'undefined')}")
        print(f"   Launcher Path: {launcher.get('path_info', 'undefined')}")
    
    # Verify Xpra configuration
    xpra_cmd = config['command']
    if '--html=on' in xpra_cmd:
        print("✅ HTML5 client enabled")
    if '--bind-tcp=' in ' '.join(xpra_cmd):
        print("✅ TCP binding configured")
    if '--daemon=no' in xpra_cmd:
        print("✅ Foreground mode enabled")
        
    print("\n🎯 Key Finding:")
    new_window = config.get('new_browser_window', True)
    if new_window is False:
        print("✅ Extension configured to NOT open new browser window")
        print("   This should work with JupyterLab tab integration!")
    else:
        print("❌ Extension may still open new browser windows")
        
except Exception as e:
    print(f"❌ Error testing server proxy: {e}")
    import traceback
    traceback.print_exc()

## Section 5: Build and Install Extension

If the extension needs to be rebuilt and reinstalled on your JupyterHub server.

In [ ]:
#!/bin/bash

# Build the extension locally (already done)
echo "🔨 Building extension..."
uv build

# Check if wheel was created
echo "📦 Checking build output..."
ls -la dist/jupyterlab_firefox_launcher-0.7.1-py3-none-any.whl

echo "✅ Extension built successfully!"
echo ""
echo "📋 Next Steps for JupyterHub Deployment:"
echo "1. Upload wheel to server:"
echo "   scp dist/jupyterlab_firefox_launcher-0.7.1-py3-none-any.whl user@your-server:/tmp/"
echo ""
echo "2. Install on server:"
echo "   sudo pip install /tmp/jupyterlab_firefox_launcher-0.7.1-py3-none-any.whl --force-reinstall"
echo ""
echo "3. Restart JupyterHub:"
echo "   sudo systemctl restart jupyterhub"
echo ""
echo "4. Test in JupyterLab:"
echo "   - Look for Firefox icon in Launcher > Other section"
echo "   - Click icon to open Firefox in JupyterLab tab"

## Section 6: Test Extension in JupyterLab

How to properly test the Firefox launcher to ensure it opens in JupyterLab tabs.

In [ ]:
#!/usr/bin/env python3
"""
Testing checklist and troubleshooting for JupyterLab tab integration
"""

print("🧪 JupyterLab Firefox Launcher Testing Guide")
print("=" * 60)

print("\n✅ CORRECT Testing Method:")
print("1. Access JupyterHub: https://your-jupyterhub-domain/")
print("2. Start your server")
print("3. Open JupyterLab interface")
print("4. Look for Firefox icon in Launcher under 'Other' section")
print("5. Click the Firefox icon")
print("6. Firefox should open in a NEW JUPYTERLAB TAB")

print("\n❌ INCORRECT Testing Method:")
print("1. Directly accessing: https://your-domain/user/ubuntu/a/firefox-desktop/")
print("2. This bypasses the JupyterLab extension entirely")
print("3. Opens Firefox in a new browser tab (not what we want)")

print("\n🔍 Troubleshooting Steps:")

print("\n1. Check Extension Installation:")
print("   Run: jupyter labextension list | grep firefox")
print("   Should show: jupyterlab-firefox-launcher v0.7.1")

print("\n2. Check Browser Console:")
print("   - Open browser dev tools in JupyterLab")
print("   - Look for: 'Firefox launcher frontend component loaded'")
print("   - Look for: 'Opening Firefox desktop at: <url>'")

print("\n3. Verify URL Construction:")
print("   - Extension should use URLExt.join() for proper service prefix")
print("   - Should NOT use window.location.origin directly")

print("\n4. Check Widget Creation:")
print("   - Extension should create iframe widget")
print("   - Should call app.shell.add(widget, 'main')")
print("   - Should NOT call window.open()")

print("\n🎯 Expected Behavior:")
print("   ✅ Firefox icon appears in JupyterLab launcher")
print("   ✅ Clicking icon opens new JupyterLab tab")
print("   ✅ Tab title shows 'Firefox Desktop'")
print("   ✅ Firefox runs inside JupyterLab interface")
print("   ✅ No new browser windows/tabs open")

print("\n📝 If Problems Persist:")
print("   1. Check JupyterHub logs: sudo journalctl -u jupyterhub -f")
print("   2. Verify extension wheel was installed correctly")
print("   3. Ensure JupyterHub was restarted after installation")
print("   4. Try refreshing JupyterLab page (Ctrl+F5)")

print("\n🎉 Success Indicator:")
print("When working correctly, you'll have Firefox running INSIDE")
print("JupyterLab as a tab, not in a separate browser window!")